# 2-link arm RL visualization

상태는

$$
x=(q_1,q_2,\dot q_1,\dot q_2)\in\mathbb{R}^4
$$

이고 제어입력은 두 관절 torque

$$
u=(\tau_1,\tau_2)\in\mathbb{R}^2
$$

입니다.

동역학은

$$
M(q)\ddot q+C(q,\dot q)\dot q+g(q)+B\dot q=\tau
$$

를 사용합니다.

PyBullet은 Colab에서 DIRECT mode renderer로 사용하고, 같은 rollout을 $(q_1,q_2)$, $(q_1,\dot q_1)$, $(q_2,\dot q_2)$ 상태공간과 동기화해 MP4로 표시합니다.

세부 식과 코드 구성은 `docs/two_link_arm.md`를 참고합니다.

## CEM

정책 parameter를 직접 샘플링하고 높은 return의 elite 집합으로 sampling distribution을 갱신합니다.

In [ ]:
!pip -q install pybullet gymnasium stable-baselines3 imageio imageio-ffmpeg
!git clone -q https://github.com/HisameOgasahara/RL_tmp.git
%cd RL_tmp

import sys
sys.path.insert(0, ".")

from IPython.display import Video, display
import matplotlib.pyplot as plt


In [ ]:
from src.rl_tmp.two_link_arm.env import TwoLinkArmEnv

env_kwargs = {
    "dt": 0.04,
    "horizon": 150,
    "target": (1.25, 1.0),
    "initial_state": (0.0, 0.0, 0.0, 0.0),
    "random_reset": True,
}


In [ ]:
from src.rl_tmp.two_link_arm.cem import train_cem

result = train_cem(
    env_kwargs=env_kwargs,
    iterations=35,
    population_size=192,
    elite_fraction=0.12,
    seed=0,
)

plt.figure(figsize=(7, 4))
plt.plot(result.history["elite_mean_reward"], label="elite mean reward")
plt.plot(result.history["best_reward"], label="best reward")
plt.xlabel("CEM iteration")
plt.ylabel("episode return")
plt.legend()
plt.show()


In [ ]:
from src.rl_tmp.two_link_arm.cem import CEMPolicy
from src.rl_tmp.two_link_arm.render import rollout_policy, render_rollout_video

for name in ["early", "middle", "final"]:
    env = TwoLinkArmEnv(**env_kwargs)
    policy = CEMPolicy(result.checkpoints[name])
    rollout = rollout_policy(policy, env, seed=123)
    path = render_rollout_video(
        rollout,
        f"/content/two_link_cem_{name}.mp4",
        config=env.config,
        fps=25,
        frame_stride=2,
    )
    print(name)
    display(Video(str(path), embed=True))
